# Phase 2 — Feature Engineering

**Goal of this notebook:** understand what we have, what needs fixing, and what we plan to build — before writing any transformation code.

---

## 1. Environment

Run this from the repo root with the project venv active (`.venv`).
Data lives in `titanic/data/raw/` — download it first if needed:

```bash
cd titanic && python setup_kaggle.py
```

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path('../data/raw')

train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')

print(f'train: {train.shape}   test: {test.shape}')

## 2. What we have

A quick look at types, nulls, and a sample row — nothing more.

In [ ]:
train.info()

In [ ]:
# Missing values in both splits — important to check test too, not just train
missing = pd.DataFrame({
    'train_missing': train.isnull().sum(),
    'test_missing':  test.isnull().sum(),
})
missing[missing.sum(axis=1) > 0]

## 3. Feature inventory

| Feature | Type | Missing | What to do |
|---|---|---|---|
| `Survived` | target (0/1) | — | keep as-is — **do not use as input to any imputation or feature** (target leakage) |
| `Pclass` | ordinal (1/2/3) | none | keep as-is |
| `Name` | string | none | **extract Title** (Mr, Mrs, Miss, Master, Rare) — strong survival signal and useful for age imputation |
| `Sex` | binary string | none | encode → 0/1 |
| `Age` | float | ~20% | **impute via multiple linear regression** using Pclass, Sex, Fare, Title — fit on rows with known age, predict missing; better than median because it borrows signal from all features simultaneously rather than relying on sparse manual subsets |
| `SibSp` | int | none | combine with Parch → **FamilySize** = SibSp + Parch + 1; also flag solo travellers |
| `Parch` | int | none | (see SibSp) |
| `Ticket` | string | none | mostly noise; can drop or extract ticket prefix — low priority |
| `Fare` | float | 1 in test | **bin into quartiles** (cheap/mid/expensive/luxury); correlated with Pclass so may add little |
| `Cabin` | string | ~77% | **extract deck letter** (A–G, T) where present; fill rest as 'U' (unknown) — too sparse to use in age imputation |
| `Embarked` | string | 2 in train | fill with mode ('S'); likely drop after — it's a proxy for class, not a causal feature |

**Features that will be dropped before modelling:** `PassengerId`, `Name` (after title extraction), `Ticket`, possibly `Cabin` if deck adds nothing.

**Order matters:** extract Title from Name *before* imputing Age — the regression needs Title as an input.

## 3a. Age imputation — why regression, not median

**The problem with median (even segmented median):** to use known relationships (older in 1st class, Masters are children, solo women in 1st class skew old) you'd need to slice by Pclass × Title × Sex × Fare simultaneously. Those buckets go sparse fast — a few passengers per cell — and a mean/median of 3 people is noise.

**What multiple linear regression buys you:** fit one model on all rows with known Age using `Pclass`, `Sex`, `Fare`, and extracted `Title` as inputs. The model learns a coefficient for each feature from *all* available data at once, then predicts the missing ages by plugging in each passenger's known values. Sparse buckets stop being a problem because the model borrows strength globally.

**The imputation pipeline (conceptual):**
```
1. Extract Title from Name  ← must happen first
2. Encode categorical inputs (Title, Sex) numerically
3. Fit LinearRegression on rows where Age is not null
4. Predict Age for rows where Age is null
5. Clip predictions to a sensible range (e.g. 0.5–80)
```

**Target leakage reminder:** `Survived` is not available in the test set at inference time — never use it as a feature, including inside imputation.

## 5. Feature Engineering

### 5a. Title extraction

Extract title from Name, then collapse into 5 categories: `Mr`, `Mrs`, `Miss`, `Master`, `Rare`.

**Why these 5:**
- `Master` — male child, median age 4, very tight distribution — almost a perfect age predictor alone
- `Miss` — unmarried female, median age 22, reasonably tight
- `Mrs` — married female, median age 35.5
- `Mr` — widest age range (15 to 80+), needs other features to impute well
- `Rare` — Dr, Rev, Col, Dona, etc. — median age 44.5, grouped to avoid sparse categories

**Order:** title extraction must happen before age imputation — Title is an input to the regression.

In [ ]:
def extract_title(name_series):
    return name_series.str.split(',', n=1).str[1].str.strip().str.split('.', n=1).str[0]

TITLE_MAP = {
    'Mr':     'Mr',
    'Mrs':    'Mrs',
    'Miss':   'Miss',
    'Master': 'Master',
}

# Work on combined train+test so both get identical treatment
df = pd.concat([train, test]).reset_index(drop=True)
df['Title'] = extract_title(df['Name']).map(TITLE_MAP).fillna('Rare')

df['Title'].value_counts()

In [ ]:
# Validate: age distribution and missing rate per title
total   = df.groupby('Title').size()
missing = df[df['Age'].isna()].groupby('Title').size()

pd.DataFrame({
    'count':        total,
    'missing':      missing,
    'missing_%':    (missing / total * 100).round(1),
    'median_age':   df.groupby('Title')['Age'].median(),
}).sort_values('median_age')

### 5b. Fare imputation

One passenger in the test set is missing Fare. Fill with the median fare for their Pclass — better than the global median since fare varied significantly by class.

In [ ]:
df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))

print(f"Missing Fare remaining: {df['Fare'].isna().sum()}")

### 5c. Encoding categorical features

- `Sex` — binary → map to 0/1 (no dummy trap risk)
- `Title` — 5 unordered categories → one-hot encoding, drop `Mr` as reference (most common, clearest meaning)

In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

title_dummies = pd.get_dummies(df['Title']).drop(columns='Mr')

print("Sex encoding:"); print(df['Sex'].value_counts())
print("\nTitle dummies:"); print(title_dummies.sum())

### 5d. Age imputation via multiple linear regression

Features used: `Pclass`, `Sex`, `Fare`, and title dummies (Master, Miss, Mrs, Rare — Mr is the reference).

Regression predicts the *conditional mean* age, so predicted values will cluster more tightly than observed ages — lower std and max are expected, not a bug. The key sanity check is that the median is similar between known and predicted.

In [ ]:
from sklearn.linear_model import LinearRegression

# Assemble feature matrix
features = pd.concat([
    df['Pclass'],
    df['Sex'],
    df['Fare'],
    title_dummies,
], axis=1)

# Split into known and unknown age
known   = features[df['Age'].notna()]
unknown = features[df['Age'].isna()]
y_known = df.loc[df['Age'].notna(), 'Age']

# Fit on known, predict on unknown
model = LinearRegression()
model.fit(known, y_known)
predicted = model.predict(unknown).clip(0.5, 80)

# Fill missing ages
df.loc[df['Age'].isna(), 'Age'] = predicted

print(f"Missing ages remaining: {df['Age'].isna().sum()}")

# Sanity check: known vs predicted distributions
pd.DataFrame({
    'known':     y_known.describe(),
    'predicted': pd.Series(predicted).describe(),
}).round(1)

### 5e. Family size and solo traveller flag

`FamilySize` = SibSp + Parch + 1 (the passenger counts themselves).
`is_solo` = 1 when travelling alone.

These replace SibSp and Parch individually — family size as a whole is more predictive than its components separately. Note that `is_solo` is confounded by class and gender (solo 3rd-class men vs solo 1st-class women have opposite survival profiles), so its value comes from tree-based models that can discover those interactions automatically.

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['is_solo']    = (df['FamilySize'] == 1).astype(int)

df['FamilySize'].value_counts().sort_index()

### 5f. Cabin → Deck

Extract the deck letter (first character of Cabin). Fill missing and the single erroneous `T` value as `U` (unknown) — 77% of passengers have no cabin recorded.

**Encoding:** one-hot, dropping `U` as the reference category (largest group). All deck coefficients are then interpreted as "relative to an unknown-deck passenger."

**Note:** Deck order on the ship was A (top) → G (bottom), suggesting an ordinal relationship with survival. However A had mixed occupants (wealthy passengers and officers with conflicting survival incentives), so one-hot is safer — no forced ordering assumption.

In [ ]:
df['Deck'] = df['Cabin'].str[0].fillna('U').replace('T', 'U')

deck_dummies = pd.get_dummies(df['Deck'], prefix='Deck').drop(columns='Deck_U')

print(df['Deck'].value_counts())
print(f"\nDeck dummy columns: {list(deck_dummies.columns)}")

### 5g. Embarked

Two missing values in train — fill with mode (`S`, the most common port). One-hot encode dropping `S` as reference.

In [ ]:
df['Embarked'] = df['Embarked'].fillna('S')

embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Emb').drop(columns='Emb_S')

print(df['Embarked'].value_counts())
print(f"\nEmbarked dummy columns: {list(embarked_dummies.columns)}")

## 5h. Correlation analysis — feature validation

Correlation of all engineered features against `Survived` (train only — test has no Survived column).

This is a **filter step**: rank features by linear correlation before modelling. Weak correlations don't automatically mean a feature is useless (tree models discover non-linear interactions), but near-zero values with no domain justification are candidates to drop.

In [ ]:
# Assemble full feature matrix (train rows only for correlation)
df_full = pd.concat([df, title_dummies, deck_dummies, embarked_dummies], axis=1)
df_train = df_full[df_full['Survived'].notna()]

corr = (df_train
    .corr(numeric_only=True)['Survived']
    .drop('Survived')
    .sort_values()
)

# Drop columns we won't use in modelling
corr = corr.drop(['PassengerId', 'SibSp', 'Parch'], errors='ignore')

corr

## 5i. Final feature matrix

Assemble all engineered features into a clean dataframe, then split back into train and test sets ready for modelling.

**Features kept:** Pclass, Sex, Age, Fare, FamilySize, is_solo, Title dummies, Deck dummies, Embarked dummies.

**Features dropped:** PassengerId (identifier, no signal), Name (Title extracted), Ticket (high cardinality noise), Cabin (Deck extracted), SibSp and Parch (replaced by FamilySize and is_solo).

In [ ]:
FEATURE_COLS = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'is_solo']

df_model = pd.concat([
    df[FEATURE_COLS],
    title_dummies,
    deck_dummies,
    embarked_dummies,
], axis=1)

# Split back into train and test
train_mask = df['Survived'].notna()

X_train = df_model[train_mask].reset_index(drop=True)
y_train = df.loc[train_mask, 'Survived'].astype(int).reset_index(drop=True)
X_test  = df_model[~train_mask].reset_index(drop=True)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")
print(f"\nMissing values in X_train: {X_train.isna().sum().sum()}")
print(f"Missing values in X_test:  {X_test.isna().sum().sum()}")

---

## 6. Repeatable pattern for future competitions

1. **Load both splits together** — check missing values in *test* too, not just train. A feature missing in test at inference time is a silent bug.
2. **Inventory before transforming** — understand each column's type, cardinality, and null rate before writing any transformation.
3. **Engineer features before imputing** — extracted features (e.g. Title) may be inputs to imputation models.
4. **Impute with signal** — regression over median when multiple features correlate with the missing column.
5. **Encode correctly** — ordinal for ordered categories, one-hot for unordered, binary map for two-value columns.
6. **Create, then evaluate** — build candidate features, then let the model (feature importance / permutation importance) tell you which ones actually help. Don't delete features by intuition alone.
7. **Apply identical transforms to test** — all imputation and encoding must fit on train only, then transform both. Fitting on the full dataset is data leakage.
8. **Never touch the target** — `Survived` does not exist in test. Any feature derived from it is leakage.